# Multi-form RAG Quick Eval
Lightweight handcrafted eval over the sample forms (~10 Q/A pairs).
Run after `python data/generate_samples.py` and ingesting all samples.

In [ ]:
from pathlib import Path
from form_agent.agent import FormAgent
agent = FormAgent()
for p in Path('../data/samples').glob('*.pdf'):
    if not any(f.source_path.endswith(p.name) for f in agent.list_forms()):
        agent.ingest(p)
[(f.id, f.form_type) for f in agent.list_forms()]

In [ ]:
EVAL = [
    ('What is the total of all expense reports?', 'aggregate'),
    ('How many job applicants have more than 5 years of experience?', 'aggregate'),
    ('Which patient reported migraines?', 'rag'),
    ('What allergies does the patient report?', 'rag'),
    ('Who has the highest salary expectation?', 'aggregate'),
    ('Which expense report is still pending approval?', 'rag'),
    ('What was the project code for Maya Chen\'s expense report?', 'rag'),
    ('How many forms are job applications?', 'aggregate'),
    ('Summarize the medical complaints across all forms.', 'rag'),
    ('What is the average salary expectation across applicants?', 'aggregate'),
]
rows = []
for q, expected in EVAL:
    ans = agent.ask_all(q)
    rows.append({'q': q, 'expected_strategy': expected, 'got_strategy': ans.strategy,
                 'answer': ans.answer, 'confidence': ans.confidence})
import json
print(json.dumps(rows, indent=2, default=str))

In [ ]:
match = sum(r['got_strategy'] == r['expected_strategy'] for r in rows)
print(f'router agreement: {match}/{len(rows)}')